In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import lagrange
from scipy.special import jv

# Potential

https://linkinghub.elsevier.com/retrieve/pii/S0375960118309587


$$
A_z(r) = m\omega F_0 J_2(\lambda r)\\
A_{\theta} = B_z
$$

In [ ]:
def V_eff(r, p):
# parameters: par = [p_theta, p_z, omega, m, H, z_0, theta_0]    
    V = 0.5/p[3] *( p[0]/r - 0.5*p[3]*p[2]*r )*( p[0]/r - 0.5*p[3]*p[2]*r ) + 0.5/p[3] *(p[1] - eA_z(r,p)) *(p[1] - eA_z(r,p))
    return V

def eA_z(r,p):
    F_0 = 1
    lambd =7.5
    result = p[2]*p[3]*F_0*jv(2, lambd*r)
    return result

def eA_zedA_z(r,p):
    result = r*0
    return result

def r_dot(r , p):
    # parameters: par = [p_theta, p_z, omega, m, H, z_0, theta_0]    
    #result = np.sqrt(2/p[2]* p[4] - 1/p[2]**2*(p[0]/r + p[2]*p[3]/2*r)**2 - 1/p[3]**2*(p[1] + eA_z(r,p)**2 ), dtype='complex')
    result = np.sqrt(2/p[3]* p[4] - 2/p[3]* V_eff(r,p), dtype='complex' )
    return np.real(result)

# Initialization

In [ ]:
# parameters: par = [p_theta, p_z, omega, m, H, z_0, theta_0]
par = np.array(     [0.3,     0.6,  4,     1, 2, 0,   0], dtype='float')

## Potential dependences from $p_\theta$ and $p_z$
1. $p_\theta$ increase the steepning of the initial central repulsive part: $\propto (p_\theta/r - \omega r)^2$
2. $p_z$ increase the height of the double well (<i>due to the bessel function $J_2$</i>)

In [ ]:
###########################
dx = 0.001     # resolution
p_theta = np.arange(0.2, 2.1, 0.4)
p_z = np.arange(0.2, 2.1, 0.4)

print(len(p_theta))
###########################

x = np.arange(0.1, 1, dx)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

for j, p_th in enumerate(p_theta):
    par[0] = p_th
    y = V_eff(x, par)
    ax[0].plot(x,y, label= rf'$p_{{\theta}}={par[0]:.1f}$' )

ax[0].legend()
ax[0].set_ylim(0,10)
fig.suptitle(rf'Effective Potential vs $p_\theta$')
ax[0].set_xlabel('Radial distance')
ax[0].set_ylabel('Energy')

par[0] = 0.3

for j, pz in enumerate(p_z):
    par[1] = pz
    y = V_eff(x, par)
    ax[1].plot(x,y, label= rf'$p_{{z}}={par[1]:.1f}$' )

ax[1].legend()
ax[1].set_ylim(0,10)
fig.suptitle(rf'Effective Potential vs $p_z$')
ax[1].set_xlabel('Radial distance')
ax[1].set_ylabel('Energy')

par[1] = 0.6

### resolution

In [ ]:
dx = 0.001     

x = np.arange(0.0, 1.9, dx)

# Magnetic field

$$
B_{\theta} = - \partial_r A_z\\
B_z = \frac{m\omega}{e}
$$

In [ ]:
def B_th(r, p =par):
    F_0 = 1
    lambd =7.5
    return - p[2]*p[3]*F_0*0.5*(jv(1, lambd*r)-jv(3, lambd*r))*lambd

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(14,10))

ax[0,0].plot(x, B_th(x, p=par))
ax[0,0].set_xlabel('$r$ - radial direction')
ax[0,0].set_ylabel('$B_{\theta}$ - Azimuthal Magnetic Field')


Bz = np.zeros(x.shape)
Bz = Bz + par[2]*par[3]
ax[0,1].plot(x, Bz)
ax[0,1].set_xlabel('$r$ - radial direction')
ax[0,1].set_ylabel('$B_{\theta}$ - Azimuthal Magnetic Field')

# Toroidal flux
PhiT = Bz * np.pi*x**2
PhiTh = x * eA_z(x, p=par)

q = PhiTh/PhiT
ax[1,0].plot(x, q)
ax[1,0].set_xlabel('$r$ - radial direction')
ax[1,0].set_ylabel('$q$ - q-profile')

theta = np.linspace(0, 2*np.pi, 17)[:-1]
r = np.linspace(0,1,8)

R, TH = np.meshgrid(r,theta)

X = R*np.cos(TH)
Y = R*np.sin(TH)

Bth = B_th(R, p=par)
U = -np.sin(TH)*Bth
V =  np.cos(TH)*Bth

ax[1,1].quiver(X,Y,U,V)
ax[1,1].set_xlabel('$r$ - radial direction')
ax[1,1].set_ylabel('$Z$ - vertiacal direction')
ax[1,1].set_aspect('equal')

fig.suptitle('Magnetic configuration')

# The magnetic field lines

In [ ]:
theta = np.linspace(0, 2*np.pi, 1000)[:-1]
r = np.linspace(0,1,100)

R, TH = np.meshgrid(theta,r)

X = R*np.cos(TH)
Y = R*np.sin(TH)

Bth = B_th(R, p=par)
U = -np.sin(TH)*Bth
V =  np.cos(TH)*Bth

ax = plt.subplot(polar=True)
ax.streamplot(theta, r, Bth, np.zeros(Bth.shape), arrowsize=0, density=.2 )
